In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

src = spark.read.parquet(
    "abfss://basecontainer@ecommprojadls.dfs.core.windows.net/bronze/customers/")

cols = ["customer_zip_prefix", "customer_city", "customer_state", "updated_at"]

(DeltaTable.forName(spark, "ecomm.silver.customers_current").alias("t")
   .merge(src.alias("s"), "t.customer_unique_id = s.customer_unique_id")
   .whenMatchedUpdate(
       condition="s._change_version > t._change_version AND s._op != 'D'",
       set={"customer_zip_prefix": "s.customer_zip_prefix",
            "customer_city": "s.customer_city",
            "customer_state": "s.customer_state",
            "updated_at": "s.updated_at",
            "_change_version": "s._change_version",
            "_is_deleted": F.lit(False),
            "_merged_at": F.current_timestamp()}
       )
   .whenMatchedUpdate(
       condition="s._change_version > t._change_version AND s._op = 'D'",
       set={"_is_deleted": F.lit(True),
            "_change_version": "s._change_version",
            "_merged_at": F.current_timestamp()}
       )
   .whenNotMatchedInsert(values={
       "customer_unique_id": "s.customer_unique_id",
       "customer_zip_prefix": "s.customer_zip_prefix",
       "customer_city": "s.customer_city",
       "customer_state": "s.customer_state",
       "_change_version": "s._change_version",
       "_is_deleted": F.expr("s._op = 'D'"),
       "_merged_at": F.current_timestamp()}
        )
   .execute())